In [2]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [3]:
### Shortcut for package import
from pkgimp import *

from nb2p import fileop, database, config
from nb2p.notebook import Notebook
from nb2p.config import default_config as conf

In [ ]:
### Select dataset to process

DATASET_NAME = "distilkaggle"

In [6]:
DIRS = config.dirs(DATASET_NAME)
DIRS.makedirs()

In [8]:
SETUP = "slt50k40"

In [7]:
db = database.connect(dataset_name=DATASET_NAME)

In [ ]:
sample_nb_ids = fileop.read_json(DIRS.base / f"sample_nb_ids-{SETUP}.json")
print(len(sample_nb_ids), sample_nb_ids[0])

In [ ]:
notebooks_data = list(
    db.notebook.find(
        {"_id": {"$in": [ObjectId(id) for id in sample_nb_ids]}},
        {"notebook_id": 1, "user_name": 1, "current_url_slug": 1},
    )
)
print(len(notebooks_data), notebooks_data[0])

In [114]:
import shutil

DEST_DIR = DIRS.base / "test_nbs"

os.makedirs(DEST_DIR, exist_ok=True)

for nb_data in notebooks_data:
    if DATASET_NAME == "kgtorrent":
        path = (
            DIRS.notebook
            / f"{nb_data['user_name']}_{nb_data['current_url_slug']}.ipynb"
        )
    elif DATASET_NAME == "gh17":
        path = DIRS.notebook / f"nb_{nb_data['notebook_id']}.ipynb"
    else:
        break

    try:
        shutil.copy2(path, DEST_DIR)
    except Exception as e:
        print(e)